In [1]:
import pandas as pd 
import numpy as np
import sklearn
import matplotlib
import openpyxl
import os

In [2]:
df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

In [3]:
df_2 = df.copy()

# Task 1: Data Profiling and Missing Values (~25 minutes)
1.1 — Profile the raw data
Compute a comprehensive profile of the dataset:

Number of rows, columns, and memory usage
Missing value counts and percentages for each column
Number of unique values per column
Basic statistics for numeric columns (min, max, mean, median, std)

In [6]:
df.shape

(541909, 8)

In [7]:
df.memory_usage(deep=True).sum()

np.int64(132310571)

In [8]:
missing = df.isna().sum().to_frame("Missing Count")

In [9]:
missing

,Missing Count
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [10]:
perc = (missing['Missing Count'] / len(df)) * 100

In [11]:
perc

InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
Name: Missing Count, dtype: float64

In [12]:
df.nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

In [13]:
df.describe(include = {'int64', 'float64'})

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


### 1.2 — Classify the missing data types
The two columns with significant missing values are CustomerID and Description.

For each:

Determine whether the missingness is MNAR, MAR, or MCAR. Justify your answer with evidence (e.g., compare rows with and without missing values — do they differ systematically in other columns?).
Decide on a strategy: deletion, imputation, or indicator column. Justify your choice.
Guiding questions:

Are transactions with missing CustomerID different from those with a CustomerID? Check the distribution of Country, Quantity, and UnitPrice for both groups.
Do transactions with missing Description have valid StockCode values? Could you recover descriptions from other rows with the same StockCode?

In [15]:
df.loc[df['CustomerID'].isna()]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
541536,581498,85099B,JUMBO BAG RED RETROSPOT,5,2011-12-09 10:26:00,4.13,NaN,United Kingdom
541537,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,2011-12-09 10:26:00,4.13,NaN,United Kingdom
541538,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-12-09 10:26:00,4.96,NaN,United Kingdom
541539,581498,85174,S/4 CACTI CANDLES,1,2011-12-09 10:26:00,10.79,NaN,United Kingdom


In [16]:
df.loc[df['CustomerID'].isna()]['Country'].value_counts()

Country
United Kingdom    133600
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
France                66
Israel                47
Portugal              39
Bahrain                2
Name: count, dtype: int64

Many transactions have missing CustomerID, and most of them are from the United Kingdom. This suggests the missingness is related to transaction conditions (e.g., guest purchases), so it is classified as MAR.

In [18]:
df.loc[df['Description'].isna()]['Country'].value_counts()

Country
United Kingdom    1454
Name: count, dtype: int64

In [19]:
df[df["Description"].isna()][["StockCode"]].head()

,StockCode
622,22139
1970,21134
1971,22145
1972,37509
1987,85226A


In [20]:
df.groupby("StockCode")["Description"].nunique().sort_values(ascending=False).head()

StockCode
20713    8
23084    7
21830    6
85175    6
21181    5
Name: Description, dtype: int64

Missing Description values occur only in the United Kingdom, but the rows still contain valid StockCode. This suggests the missingness is related to the transaction process and can be classified as MAR.

### 1.3 — Handle the missing values
Apply your chosen strategies. Document what you did and why. After handling missing values, print the remaining missing value counts to confirm.

Deliverable for Task 1
A profiling summary, a written classification of missing data types with evidence, and clean handling with documentation.

In [23]:
df_2.dropna(subset={'CustomerID','Description'}, inplace=True)

Description There are only a few missing Description values compared to the full dataset (~541,909 rows). Dropping them has little effect on the data, so these rows are removed.

Many transactions have missing CustomerID, mostly from the United Kingdom. The missingness is related to transaction type (MAR). Since these are identifiers, rows with missing CustomerID are dropped.

In [26]:
df_2.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [27]:
df_2.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

# Task 2: Cleaning Invalid and Anomalous Records (~25 minutes)
2.1 — Identify cancellations
Invoices starting with "C" are cancellations. These have negative quantities and represent returns, not purchases.

Count how many cancellation transactions exist.
Decide whether to keep, remove, or flag them. Think about the downstream task: if you later want to predict customer churn or build a recommender, how do cancellations affect the analysis?

In [29]:
cancellations = df_2[df_2["InvoiceNo"].astype(str).str.startswith("C")]
len(cancellations)

8905

In [30]:
df_2["IsCancellation"] = df_2["InvoiceNo"].astype(str).str.startswith("C", na=False)

In [31]:
df_2["IsCancellation"].sum()

np.int64(8905)

Invoices starting with "C" represent cancellation transactions where items were returned. These records may indicate customer dissatisfaction and can be informative for tasks such as churn prediction. Therefore, instead of removing them immediately, they are flagged using a binary variable so they can be excluded or used depending on the downstream task.

### 2.2 — Handle invalid quantities and prices
Investigate records with:

Quantity <= 0 (that are not cancellations)
UnitPrice <= 0
Extreme outliers in Quantity or UnitPrice
For each category of invalid record:

Count how many exist
Decide what to do (remove, cap, flag) and justify


In [34]:
invalid_quantity = df_2[
    (df_2["Quantity"] <= 0) &
    (~df_2["InvoiceNo"].str.startswith("C", na=False))
]

invalid_quantity.shape[0]

0

Records with Quantity ≤ 0 were investigated to detect invalid transactions. The analysis shows that there are 0 records where Quantity ≤ 0 that are not cancellation invoices. This indicates that all negative quantities correspond to legitimate cancellation transactions, so no additional cleaning is required for this category.

In [36]:
invalid_price = df_2[df_2["UnitPrice"] <= 0]
invalid_price.shape[0]

40

In [37]:
df_2 = df_2[df_2["UnitPrice"] > 0]

The dataset was inspected for records where UnitPrice ≤ 0. A total of 40 records were found. These entries do not represent valid sales transactions and may correspond to data errors or promotional adjustments. To ensure accurate analysis of customer purchasing behavior and revenue, these records were removed from the dataset.

In [39]:
df_2[["Quantity","UnitPrice"]].describe()

,Quantity,UnitPrice
count,406789.000000,406789.000000
mean,12.028359,3.460811
std,247.927842,69.318561
min,-80995.000000,0.001000
25%,2.000000,1.250000
50%,5.000000,1.950000
75%,12.000000,3.750000
max,80995.000000,38970.000000


In [40]:
df_2.nlargest(10, "Quantity")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,False
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,False
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901.0,United Kingdom,False
206121,554868,22197,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,0.72,13135.0,United Kingdom,False
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,0.82,18087.0,United Kingdom,False
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,2011-07-19 17:04:00,0.06,14609.0,United Kingdom,False
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom,False
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom,False
433788,573995,16014,SMALL CHINESE STYLE SCISSOR,3000,2011-11-02 11:24:00,0.32,16308.0,United Kingdom,False
4945,536830,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,2880,2010-12-02 17:38:00,0.18,16754.0,United Kingdom,False


In [41]:
df_2.nlargest(10, "UnitPrice")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation
222681,C556445,M,Manual,-1,2011-06-10 15:31:00,38970.00,15098.0,United Kingdom,True
173277,C551685,POST,POSTAGE,-1,2011-05-03 12:51:00,8142.75,16029.0,United Kingdom,True
173382,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom,False
173391,C551699,M,Manual,-1,2011-05-03 14:12:00,6930.00,16029.0,United Kingdom,True
268027,C560372,M,Manual,-1,2011-07-18 12:26:00,4287.63,17448.0,United Kingdom,True
422351,573077,M,Manual,1,2011-10-27 14:13:00,4161.06,12536.0,France,False
422375,C573079,M,Manual,-2,2011-10-27 14:15:00,4161.06,12536.0,France,True
422376,573080,M,Manual,1,2011-10-27 14:20:00,4161.06,12536.0,France,False
406404,C571750,M,Manual,-1,2011-10-19 11:16:00,3949.32,12744.0,Singapore,True
406406,571751,M,Manual,1,2011-10-19 11:18:00,3949.32,12744.0,Singapore,False


In [42]:
q99 = df_2["Quantity"].quantile(0.99)
p99 = df_2["UnitPrice"].quantile(0.99)

extreme_quantity = df_2[df_2["Quantity"] > q99]
extreme_price = df_2[df_2["UnitPrice"] > p99]

extreme_quantity.shape[0], extreme_price.shape[0]

(3890, 3868)

In [43]:
df_2["Quantity"] = df_2["Quantity"].clip(upper=q99)
df_2["UnitPrice"] = df_2["UnitPrice"].clip(upper=p99)

Extreme values were investigated in the Quantity and UnitPrice columns. Using the 99th percentile as a threshold, 3890 records were identified as extreme quantities and 3868 records as extreme prices. These values may correspond to bulk purchases or anomalies. Instead of removing them, the values were capped at the 99th percentile to reduce their influence on statistical analysis and machine learning models.

### 2.3 — Clean and validate
Apply your cleaning rules. After cleaning, verify:

No remaining negative quantities (unless you kept cancellations with a flag)
No zero or negative prices
Print the shape before and after cleaning
Deliverable for Task 2
A documented cleaning pipeline with before/after counts, justifications for each decision, and validation checks.



In [46]:
print("Negative quantities (not cancellations):", ((df_2["Quantity"] <= 0) & (~df_2["IsCancellation"])).sum())
print("Zero or negative prices:", (df_2["UnitPrice"] <= 0).sum())
print("Shape after cleaning:", df_2.shape)

Negative quantities (not cancellations): 0
Zero or negative prices: 0
Shape after cleaning: (406789, 9)


-----------------------------------------------------------------------------

Cleaning pipeline applied:

Flagged cancellation invoices (InvoiceNo starting with "C") to preserve for downstream churn analysis.

Removed records with UnitPrice ≤ 0 (40 rows) because they do not represent valid transactions.

Verified that all negative quantities correspond to cancellations; none removed.

Extreme values in Quantity and UnitPrice were capped at the 99th percentile to prevent distortion in analysis or modeling.

Validated that no remaining invalid quantities or prices exist.

----------------------------------------------------------

Task 3: Categorical Data Challenges
3.1 — Analyze the Country column
How many unique countries are there?
What percentage of transactions come from the top 5 countries?
How many countries have fewer than 50 transactions?
Create a cleaned version of the Country column that groups rare countries (fewer than 50 transactions) into an "Other" category. Compare the number of categories before and after.

3.2 — Analyze the StockCode column
StockCode is a high-cardinality categorical feature.

How many unique stock codes exist?
Are there non-product codes (e.g., postage, discounts, manual adjustments)? Identify them by looking at codes that are not purely numeric or have unusual patterns.
Decide how to handle non-product codes for a product-level analysis.
3.3 — Engineer a feature from Description
The Description column contains free text. Create a simple feature from it — for example, the word count of the description, or a flag for descriptions containing certain keywords (e.g., "SET", "PACK", "VINTAGE"). Show that your engineered feature has a meaningful relationship with Quantity or UnitPrice.

Deliverable for Task 3
Analysis of categorical columns, a strategy for rare and high-cardinality categories, and one engineered feature with evidence of its usefulness.

# Task 3: Categorical Data Challenges 
### 3.1 — Analyze the Country column How many unique countries are there? What percentage of transactions come from the top 5 countries? How many countries have fewer than 50 transactions? Create a cleaned version of the Country column that groups rare countries (fewer than 50 transactions) into an "Other" category. Compare the number of categories before and after.

In [52]:
num_countries = df_2["Country"].nunique()
print("Number of unique countries:", num_countries)

Number of unique countries: 37


In [53]:
top5 = df_2["Country"].value_counts().head(5)
top5_percentage = top5.sum() / len(df_2) * 100
print("Top 5 countries:\n", top5)
print(f"Percentage of transactions from top 5 countries: {top5_percentage:.2f}%")

Top 5 countries:
 Country
United Kingdom    361854
Germany             9493
France              8490
EIRE                7483
Spain               2532
Name: count, dtype: int64
Percentage of transactions from top 5 countries: 95.84%


In [54]:
rare_countries = df_2["Country"].value_counts()[df_2["Country"].value_counts() < 50]
print("Number of countries with <50 transactions:", rare_countries.shape[0])

Number of countries with <50 transactions: 6


In [55]:
rare_countries

Country
Lebanon           45
Lithuania         35
Brazil            32
Czech Republic    30
Bahrain           17
Saudi Arabia      10
Name: count, dtype: int64

In [90]:
country_counts = df_2["Country"].value_counts()

In [92]:
df_2["Country_Cleaned"] = df_2["Country"].where(df_2["Country"].map(country_counts) >= 50, "Other")

In [96]:
print("Number of categories before:", df_2["Country"].nunique())
print("Number of categories after cleaning:", df_2["Country_Cleaned"].nunique())

Number of categories before: 37
Number of categories after cleaning: 32


### 3.2 — Analyze the StockCode column 
StockCode is a high-cardinality categorical feature.

How many unique stock codes exist? Are there non-product codes (e.g., postage, discounts, manual adjustments)? Identify them by looking at codes that are not purely numeric or have unusual patterns. Decide how to handle non-product codes for a product-level analysis.

In [98]:
num_stockcodes = df_2["StockCode"].nunique()
print("Number of unique StockCodes:", num_stockcodes)

Number of unique StockCodes: 3684


In [112]:
non_product_codes = df_2[~df_2["StockCode"].astype(str).str.contains(r'\d')]["StockCode"].unique()
print(non_product_codes)

['POST' 'D' 'M' 'BANK CHARGES' 'PADS' 'DOT' 'CRUK']


In [117]:
product_df = df_2[~df_2["StockCode"].isin(non_product_codes)]

These codes represent postage, bank charges, or adjustments, not actual products. For product-level analysis (like sales per item or stock-level patterns), they are removed so that only real products are included.

### 3.3 — Engineer a feature from Description 
The Description column contains free text. Create a simple feature from it — for example, the word count of the description, or a flag for descriptions containing certain keywords (e.g., "SET", "PACK", "VINTAGE"). Show that your engineered feature has a meaningful relationship with Quantity or UnitPrice.

Deliverable for Task 3 Analysis of categorical columns, a strategy for rare and high-cardinality categories, and one engineered feature with evidence of its usefulness.

In [133]:
# Create a word count feature
df_2["Desc_WordCount"] = df_2["Description"].astype(str).str.split().str.len()
# Quick check
df_2[["Description", "Desc_WordCount", "Quantity", "UnitPrice"]].head()

,Description,Desc_WordCount,Quantity,UnitPrice
0,WHITE HANGING HEART T-LIGHT HOLDER,5,6,2.55
1,WHITE METAL LANTERN,3,6,3.39
2,CREAM CUPID HEARTS COAT HANGER,5,8,2.75
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,6,3.39
4,RED WOOLLY HOTTIE WHITE HEART.,5,6,3.39


In [137]:
keywords = ["SET", "PACK", "VINTAGE"]
df_2["Contains_Keyword"] = df_2["Description"].astype(str).str.upper().apply(
    lambda x: any(k in x for k in keywords)
)

In [135]:
# Show only rows where Contains_Keyword is True
df_2[df_2["Contains_Keyword"]][["Description", "Contains_Keyword", "Quantity", "UnitPrice"]].head()

,Description,Contains_Keyword,Quantity,UnitPrice
5,SET 7 BABUSHKA NESTING BOXES,True,2,7.65
15,BOX OF VINTAGE JIGSAW BLOCKS,True,3,4.95
16,BOX OF VINTAGE ALPHABET BLOCKS,True,2,9.95
21,JAM MAKING SET WITH JARS,True,6,4.25
32,VINTAGE HEADS AND TAILS CARD GAME,True,24,1.25


In [123]:
df_2.groupby("Contains_Keyword")["UnitPrice"].mean()

Contains_Keyword
False    2.902927
True     2.688044
Name: UnitPrice, dtype: float64

In [127]:
df_2.groupby(pd.cut(df_2["Desc_WordCount"], bins=[0,1,3,5,10,50]))["UnitPrice"].mean()

C:\Users\royan\AppData\Local\Temp\ipykernel_5448\2194146939.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_2.groupby(pd.cut(df_2["Desc_WordCount"], bins=[0,1,3,5,10,50]))["UnitPrice"].mean()


Desc_WordCount
(0, 1]      12.226592
(1, 3]       2.783507
(3, 5]       2.850717
(5, 10]      2.697345
(10, 50]          NaN
Name: UnitPrice, dtype: float64

# Task 4: Class Imbalance — Predicting High-Value Customers
### 4.1 — Engineer a binary target
Create a customer-level dataset by aggregating transactions per CustomerID. Compute:

Total revenue (Quantity * UnitPrice)
Number of orders (unique InvoiceNo values)
Number of distinct products purchased
Date of first and last purchase
Define a binary target: a customer is high-value if their total revenue is in the top 10%. Label these as 1 and the rest as 0.

In [148]:
# Compute revenue per transaction
df_2["Revenue"] = df_2["Quantity"] * df_2["UnitPrice"]

# Aggregate per customer
customer_df = df_2.groupby("CustomerID").agg(
    total_revenue=("Revenue", "sum"),
    num_orders=("InvoiceNo", "nunique"),
    num_products=("Description", "nunique"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

In [150]:
threshold = customer_df["total_revenue"].quantile(0.9)
# Binary target
customer_df["high_value"] = (customer_df["total_revenue"] >= threshold).astype(int)

### 4.2 — Measure the imbalance
What is the class distribution (high-value vs. regular)?
If a model always predicted "regular," what would its accuracy be?
Why is accuracy a poor metric here?

In [156]:
customer_df["high_value"].value_counts(normalize=True)

high_value
0    0.899794
1    0.100206
Name: proportion, dtype: float64

The dataset is highly imbalanced: ~90% of customers are regular and ~10% are high-value. A model that always predicts “regular” would achieve ~90% accuracy, but it would fail to identify any high-value customers. Therefore, accuracy is misleading, and metrics like precision, recall, F1-score, or ROC-AUC should be used to properly evaluate model performance.

### 4.3 — Apply resampling
Split the customer-level dataset into train and test sets (80/20). Then apply two resampling techniques to the training set only:

Random oversampling of the minority class
Random undersampling of the majority class
For each:
Print the class distribution before and after resampling
Train a simple model (e.g., LogisticRegression or DecisionTreeClassifier) on both the original and resampled training sets

In [162]:
from sklearn.model_selection import train_test_split

X = customer_df[["total_revenue", "num_orders", "num_products"]]
y = customer_df["high_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [164]:
print("Original training set class distribution:")
print(y_train.value_counts())

Original training set class distribution:
high_value
0    3146
1     350
Name: count, dtype: int64


In [170]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

print("After oversampling:")
print(pd.Series(y_train_over).value_counts())

After oversampling:
high_value
0    3146
1    3146
Name: count, dtype: int64


In [172]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("After undersampling:")
print(pd.Series(y_train_under).value_counts())

After undersampling:
high_value
0    350
1    350
Name: count, dtype: int64


In [174]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Original
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Original training set:")
print(classification_report(y_test, y_pred))

# Oversampled
model.fit(X_train_over, y_train_over)
y_pred_over = model.predict(X_test)
print("Oversampled training set:")
print(classification_report(y_test, y_pred_over))

# Undersampled
model.fit(X_train_under, y_train_under)
y_pred_under = model.predict(X_test)
print("Undersampled training set:")
print(classification_report(y_test, y_pred_under))

Original training set:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       787
           1       1.00      1.00      1.00        88

    accuracy                           1.00       875
   macro avg       1.00      1.00      1.00       875
weighted avg       1.00      1.00      1.00       875

Oversampled training set:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       787
           1       1.00      1.00      1.00        88

    accuracy                           1.00       875
   macro avg       1.00      1.00      1.00       875
weighted avg       1.00      1.00      1.00       875

Undersampled training set:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       787
           1       0.99      1.00      0.99        88

    accuracy                           1.00       875
   macro avg       0.99      1.00      1.00       87

# Task 5: Data Leakage — Introduce, Detect, and Fix
### 5.1 — Intentionally introduce temporal leakage
The dataset spans December 2010 through December 2011. Suppose you want to predict whether a customer will make a purchase in December 2011 based on their behavior in earlier months.

First, do it wrong: randomly split all customer features (computed from the full date range) into train and test sets. Train a model and record its performance.

In [177]:
# Target: did customer purchase in Dec 2011?
df_2['InvoiceMonth'] = df_2['InvoiceDate'].dt.to_period('M')
dec_2011 = df_2[df_2['InvoiceMonth'] == '2011-12']
customers_dec = dec_2011['CustomerID'].unique()

customer_df['purchase_dec2011'] = customer_df['CustomerID'].apply(
    lambda x: 1 if x in customers_dec else 0
)

In [187]:
X = customer_df[["total_revenue", "num_orders", "num_products"]]
y = customer_df["purchase_dec2011"]

# Random split ignores time → leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train a model
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Evaluate
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.98      0.93       746
           1       0.70      0.29      0.41       129

    accuracy                           0.88       875
   macro avg       0.79      0.63      0.67       875
weighted avg       0.86      0.88      0.85       875



### 5.2 — Detect the leakage
Check whether train and test sets contain features computed from overlapping time periods.
Look for suspiciously high model performance (a sign of leakage).
Compute feature-target correlations and identify any that seem too good to be true.

In [191]:
# Are features computed from overlapping months?
print(df_2['InvoiceDate'].min(), df_2['InvoiceDate'].max())

2010-12-01 08:26:00 2011-12-09 12:50:00


In [193]:
corr = customer_df.corr()['purchase_dec2011'].sort_values(ascending=False)
print(corr)

purchase_dec2011    1.000000
last_purchase       0.375759
num_orders          0.345436
high_value          0.300070
num_products        0.287295
total_revenue       0.181460
CustomerID         -0.020653
first_purchase     -0.057450
Name: purchase_dec2011, dtype: float64


The high correlations are a classic sign of temporal leakage. Features that include the prediction period give the model unfair information.

### 5.3 — Fix with a correct temporal split
Now do it right:

Use data from December 2010 through September 2011 to compute customer features (the "observation window").
Use data from October 2011 through December 2011 to create the target variable: did the customer make at least one purchase in this "prediction window"?
Train the same model and compare performance.

In [197]:
observation_end = pd.Timestamp("2011-09-30") 
prediction_start = pd.Timestamp("2011-10-01") 
df_obs = df_2[df_2["InvoiceDate"] <= observation_end] 
df_pred = df_2[df_2["InvoiceDate"] >= prediction_start]

In [199]:
customer_features = df_obs.groupby("CustomerID").agg(
    total_revenue=("Quantity", lambda x: (x * df_obs.loc[x.index, "UnitPrice"]).sum()),
    num_orders=("InvoiceNo", "nunique"),
    num_products=("Description", "nunique"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

In [201]:
# Customers who purchased in the prediction window
customers_pred = df_pred["CustomerID"].unique()
customer_features["purchase_oct_dec"] = customer_features["CustomerID"].apply(
    lambda x: 1 if x in customers_pred else 0
)

In [203]:
X = customer_features[["total_revenue", "num_orders", "num_products"]]
y = customer_features["purchase_oct_dec"]

In [205]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.63      0.77      0.69       350
           1       0.73      0.59      0.65       380

    accuracy                           0.67       730
   macro avg       0.68      0.68      0.67       730
weighted avg       0.68      0.67      0.67       730



A temporal split is correct because to predict future purchases, features must be computed only from past data. Using all data, including the prediction period, causes temporal leakage and inflates model performance. By splitting the data into an observation window and a separate prediction window, the model learns truly predictive patterns and its performance reflects realistic future outcomes.